## Grad-CAM

### Grad-CAM Heatmap (PyTorch – Standard)

In [1]:
import torch
import torchvision.models as models
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import cv2
import numpy as np
from PIL import Image

# Load model
model = models.resnet50(pretrained=True)
model.eval()

# Load image
img = Image.open("image.jpg").convert("RGB")
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])
x = transform(img).unsqueeze(0)

# Hook storage
activations = {}
gradients = {}

def forward_hook(module, input, output):
    activations["value"] = output

def backward_hook(module, grad_in, grad_out):
    gradients["value"] = grad_out[0]

# Target layer
target_layer = model.layer4[-1].conv3
target_layer.register_forward_hook(forward_hook)
target_layer.register_backward_hook(backward_hook)

# Forward
output = model(x)
class_idx = output.argmax()

# Backward
model.zero_grad()
output[0, class_idx].backward()

# Grad-CAM
grads = gradients["value"]
acts = activations["value"]
weights = grads.mean(dim=(2, 3), keepdim=True)
cam = (weights * acts).sum(dim=1)
cam = torch.relu(cam)

# Normalize & resize
cam = cam[0].cpu().numpy()
cam = (cam - cam.min()) / (cam.max() - cam.min())
cam = cv2.resize(cam, (224, 224))

# Overlay
img_np = np.array(img.resize((224, 224)))
heatmap = cv2.applyColorMap(np.uint8(255 * cam), cv2.COLORMAP_JET)
overlay = cv2.addWeighted(img_np, 0.6, heatmap, 0.4, 0)

plt.imshow(overlay)
plt.axis("off")
plt.title("Grad-CAM Heatmap")
plt.show()


ModuleNotFoundError: No module named 'torchvision'

### Grad-CAM++ (Better Localization)

In [2]:
weights = (grads ** 2) / (2 * grads ** 2 + acts * grads ** 3 + 1e-8)
weights = weights.sum(dim=(2, 3), keepdim=True)
cam_pp = (weights * acts).sum(dim=1)
cam_pp = torch.relu(cam_pp)


NameError: name 'grads' is not defined

### Grad-CAM for Any CNN Layer (Reusable Function)

In [3]:
def grad_cam(model, x, target_layer, class_idx=None):
    activations, gradients = {}, {}

    def f_hook(m, i, o): activations["v"] = o
    def b_hook(m, gi, go): gradients["v"] = go[0]

    target_layer.register_forward_hook(f_hook)
    target_layer.register_backward_hook(b_hook)

    output = model(x)
    if class_idx is None:
        class_idx = output.argmax()

    model.zero_grad()
    output[0, class_idx].backward()

    grads = gradients["v"]
    acts = activations["v"]
    weights = grads.mean(dim=(2, 3), keepdim=True)
    cam = torch.relu((weights * acts).sum(dim=1))

    cam = cam[0].cpu().numpy()
    cam = (cam - cam.min()) / (cam.max() - cam.min())
    return cam


### Grad-CAM (TensorFlow / Keras)

In [4]:
import tensorflow as tf
import numpy as np
import cv2
import matplotlib.pyplot as plt

model = tf.keras.applications.ResNet50(weights="imagenet")

img = tf.keras.preprocessing.image.load_img(
    "image.jpg", target_size=(224, 224)
)
img = tf.keras.preprocessing.image.img_to_array(img)
img = tf.keras.applications.resnet.preprocess_input(img)
img = img[None, ...]

# Target layer
last_conv = model.get_layer("conv5_block3_out")
grad_model = tf.keras.models.Model(
    [model.inputs], [last_conv.output, model.output]
)

with tf.GradientTape() as tape:
    conv_out, preds = grad_model(img)
    class_idx = tf.argmax(preds[0])
    loss = preds[:, class_idx]

grads = tape.gradient(loss, conv_out)
weights = tf.reduce_mean(grads, axis=(1, 2))
cam = tf.reduce_sum(weights[:, None, None, :] * conv_out, axis=-1)
cam = tf.nn.relu(cam)[0].numpy()

cam = cv2.resize(cam, (224, 224))
cam = (cam - cam.min()) / (cam.max() - cam.min())

# Overlay
img_raw = tf.keras.preprocessing.image.load_img("image.jpg", target_size=(224, 224))
img_raw = tf.keras.preprocessing.image.img_to_array(img_raw)

heatmap = cv2.applyColorMap(np.uint8(255 * cam), cv2.COLORMAP_JET)
overlay = cv2.addWeighted(img_raw.astype(np.uint8), 0.6, heatmap, 0.4, 0)

plt.imshow(overlay)
plt.axis("off")
plt.title("Grad-CAM (TensorFlow)")
plt.show()


ModuleNotFoundError: No module named 'tensorflow'